In [2]:
from google.cloud import bigquery
from google.oauth2 import service_account

PROJECT_ID = "whiteberry-network"

credentials = service_account.Credentials.from_service_account_file(
    r"..\..\credentials\service-account.json"
)

client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

#### 1. NÚMERO DE CLIENTES POR PAÍS Y CANAL

In [26]:
query = """ 
SELECT 
    country,
    source_channel,
    COUNT(*) AS total_clientes
FROM `whiteberry-network.whiteberry.customers`
GROUP BY 
    country,
    source_channel
ORDER BY 
    country,
    total_clientes DESC;"""

df = client.query(query).to_dataframe()
print(df.head())

C:\Users\m_fon\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


   country source_channel  total_clientes
0   France        organic              33
1   France       paid_ads              33
2   France       referral              29
3   France   social_media              25
4  Germany       referral              36


#### 2. INGRESOS POR MES en curso
Obtenemos el precio por número de unidades por cada línea de pedido de los pedidos entregados, cuyo pago ha sido completado. 


In [ ]:

query = """ 
SELECT 
    FORMAT_DATE('%Y-%m', o.delivered_date) AS mes,
    SUM(oi.quantity * oi.unit_price) AS ingresos_mes
FROM `whiteberry-network.whiteberry.orders` o
JOIN `whiteberry-network.whiteberry.order_items` oi 
    ON o.order_id = oi.order_id
JOIN `whiteberry-network.whiteberry.payments` p 
    ON o.order_id = p.order_id
WHERE 
    o.status = 'delivered'
    AND p.status = 'completed'
    AND p.payment_type = 'payment'
    AND EXTRACT(MONTH FROM o.delivered_date) = EXTRACT(MONTH FROM CURRENT_DATE())
    AND EXTRACT(YEAR FROM o.delivered_date) = EXTRACT(YEAR FROM CURRENT_DATE())

GROUP BY 
    FORMAT_DATE('%Y-%m', o.delivered_date)"""

df = client.query(query).to_dataframe()
print(df.head())

       mes  ingresos_mes
0  2026-04     236039.05


C:\Users\m_fon\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


#### 3. MARGEN POR CATEGORÍA
Para calcular el margen tenemos que calcular el total de ingresos SUM(oi.quantity * oi.unit_price) y de ahí restar el total de costes SUM(oi.quantity * p.cost) AS costes.
Se calculará de pedidos que sí existen y están pagados y entregados. Para que el margen sea correcto la venta tiene que ser válida es decir_ :
- Para cada pedido tiene que tener líneas de pedido existentes (INNER JOIN order_items ) y el pedido tiene que estar realizado (WHERE o.status = 'delivered')
- Tiene que tener categorías existentes(INNER JOIN categories)
- Tiene que existir un pago (INNER JOIN payments pay) realizado (WHERE pay.status = 'completed' AND pay.payment_type = 'payment')
items que existen
Por último se agrupará por categorías

In [16]:
query = """ 
SELECT 
    c.name AS categoria,
    SUM(oi.quantity * oi.unit_price) AS ingresos,
    SUM(oi.quantity * p.cost) AS costes,
    SUM(oi.quantity * (oi.unit_price - p.cost)) AS margen
FROM `whiteberry-network.whiteberry.orders` o
INNER JOIN `whiteberry-network.whiteberry.order_items` oi 
    ON o.order_id = oi.order_id
INNER JOIN `whiteberry-network.whiteberry.products` p 
    ON oi.product_id = p.product_id
INNER JOIN `whiteberry-network.whiteberry.categories` c
    ON p.category_id = c.category_id
INNER JOIN `whiteberry-network.whiteberry.payments` pay
    ON o.order_id = pay.order_id
WHERE 
    o.status = 'delivered'
    AND pay.status = 'completed'
    AND pay.payment_type = 'payment'
GROUP BY 
    c.name
ORDER BY 
    margen DESC;"""

df = client.query(query).to_dataframe()
print(df.head())

     categoria   ingresos     costes     margen
0  Accessories  906841.01  527618.57  379222.44
1  Smartphones  605243.81  403292.63  201951.18
2       Gaming  527054.54  333583.99  193470.55
3   Components  650389.80  461384.31  189005.49
4      Laptops  663838.22  484122.05  179716.17


C:\Users\m_fon\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


#### 4. TOP 10 PRODUCTOS MÁS VENDIDOS
Calculamos el número de unidades vendidas con el sumatorio de quantity de líneas de pedido correspondientes a poductos entregados y cuyo pago ha sido completado

In [19]:
query = """ 
SELECT 
    p.product_id AS product_id,
    p.name AS producto,
    SUM(oi.quantity) AS unidades_vendidas
FROM `whiteberry-network.whiteberry.orders` o
INNER JOIN `whiteberry-network.whiteberry.order_items` oi 
    ON o.order_id = oi.order_id
INNER JOIN `whiteberry-network.whiteberry.products` p 
    ON oi.product_id = p.product_id
INNER JOIN `whiteberry-network.whiteberry.payments` pay
    ON o.order_id = pay.order_id
WHERE 
    o.status = 'delivered'
    AND pay.status = 'completed'
    AND pay.payment_type = 'payment'
GROUP BY 
    p.product_id, p.name
ORDER BY 
    unidades_vendidas DESC
LIMIT 10;"""

df = client.query(query).to_dataframe()
print(df.head())

   product_id                producto  unidades_vendidas
0          77  Asus Bluetooth Speaker                 86
1          55           Lenovo Webcam                 83
2          56              Sony Mouse                 81
3          44         Asus 4K Display                 77
4          62     MSI Gaming Keyboard                 72


C:\Users\m_fon\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


#### 5. RATING MEDIO POR CATEGORÍA O PRODUCTO
Con la función AVG calculamos el promedio de los valores de la columna rating de reviews agrupando por categoría

In [28]:

query = """ 
SELECT 
    c.name AS categoria,
    AVG(r.rating) AS rating_medio
FROM `whiteberry-network.whiteberry.reviews` r
INNER JOIN `whiteberry-network.whiteberry.order_items` oi 
    ON r.order_item_id = oi.order_id
INNER JOIN `whiteberry-network.whiteberry.products` p 
    ON oi.product_id = p.product_id
INNER JOIN `whiteberry-network.whiteberry.categories` c
    ON p.category_id = c.category_id
GROUP BY 
    c.name
ORDER BY 
    rating_medio DESC"""

df = client.query(query).to_dataframe()
print(df.head())

C:\Users\m_fon\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


     categoria  rating_medio
0       Gaming      4.000000
1     Monitors      3.988889
2   Components      3.838384
3        Audio      3.821429
4  Accessories      3.766990
